# Delivery times of three apps: stem-and-leaf display

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Stem-and-Leaf Display** · a worked example from the StatNex teaching guide

**The story.** The table holds the delivery times in minutes of 50 orders from each of three food-delivery apps: ZipEats, FeastNow and UrbanBite. A stem-and-leaf display keeps every individual minute visible, which a histogram does not.

**The question.** Can we see both the shape of each app's delivery times and the exact values that make it up?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `01_basic/stem_and_leaf.csv` (50 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
No.,ZipEats,FeastNow,UrbanBite
1.0,16.0,32.0,36.0
2.0,24.0,26.0,41.0
3.0,28.0,31.0,38.0
4.0,19.0,33.0,48.0
5.0,29.0,25.0,33.0
6.0,26.0,37.0,42.0
7.0,22.0,30.0,42.0
8.0,30.0,28.0,42.0
9.0,22.0,29.0,34.0
10.0,26.0,35.0,46.0
11.0,28.0,33.0,38.0
12.0,17.0,27.0,53.0
13.0,29.0,34.0,41.0
14.0,26.0,26.0,45.0
15.0,23.0,23.0,35.0
16.0,23.0,30.0,46.0
17.0,23.0,30.0,47.0
18.0,22.0,33.0,33.0
19.0,26.0,35.0,44.0
20.0,12.0,29.0,31.0
21.0,26.0,29.0,32.0
22.0,14.0,42.0,44.0
23.0,29.0,33.0,42.0
24.0,20.0,32.0,39.0
25.0,23.0,32.0,41.0
26.0,20.0,41.0,38.0
27.0,27.0,29.0,46.0
28.0,23.0,36.0,38.0
29.0,28.0,34.0,36.0
30.0,19.0,33.0,31.0
31.0,27.0,30.0,36.0
32.0,27.0,32.0,38.0
33.0,29.0,38.0,32.0
34.0,24.0,35.0,46.0
35.0,26.0,41.0,45.0
36.0,18.0,31.0,38.0
37.0,25.0,37.0,44.0
38.0,58.0,37.0,34.0
39.0,17.0,33.0,32.0
40.0,31.0,31.0,29.0
41.0,27.0,40.0,45.0
42.0,18.0,39.0,40.0
43.0,24.0,35.0,37.0
44.0,29.0,37.0,42.0
45.0,25.0,35.0,43.0
46.0,24.0,35.0,36.0
47.0,25.0,31.0,39.0
48.0,23.0,33.0,40.0
49.0,20.0,37.0,46.0
50.0,26.0,33.0,44.0
""")

## Libraries

In [ ]:
from decimal import Decimal

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
# One column per variable, sorted. Blank and text cells are left out.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["ZipEats", "FeastNow", "UrbanBite"]
series = {c: np.sort(pd.to_numeric(data[c], errors="coerce").dropna().to_numpy(float))
          for c in columns}

## Descriptive statistics

In [ ]:
# Variance and SD divide by n - 1.
summary = pd.DataFrame({
    name: {"n": x.size, "mean": np.mean(x), "variance": np.var(x, ddof=1),
           "sd": np.std(x, ddof=1), "min": x.min(), "median": np.median(x), "max": x.max()}
    for name, x in series.items()
})
print(summary)

## Stem width

In [ ]:
def stem_width(x):
    return float("5")

## Stem-and-leaf display

In [ ]:
# How to read a row: the stem gives the leading digits and each leaf one more digit,
# so stem 20 with leaves 5 5 6 means the values 25, 25 and 26. The row's length is a
# bar of a histogram, but every value can still be read off. Keep the empty rows: a
# run of them is the gap that sets an outlier apart. No standard Python function
# draws this display, so it is written out.
# The stem unit u is the power of ten at or above the width, so a width of 0.5 gives
# each whole-number stem two rows (leaves 0-4 and 5-9). A value is placed by its size
# |x|: the stem is |x| cut down to a multiple of u, and the leaf is the next digit,
# cut off rather than rounded, as in Tukey's displays: 3.97 is stem 3, leaf 9. A
# negative value takes the stem towards zero with a minus sign: -4.9 is stem -4, leaf 9,
# and a value between -1 and 0 sits on the stem -0. Each row holds the sizes in
# [lower, lower + width).
# Every bound is rounded to the decimals the width is written with, as the classes of
# a frequency table are: floating point makes 0.1 × 3 0.30000000000000004 and 0.3 / 0.1
# 2.9999999999999996, which would put a value of 0.3 a row or a leaf too low.
def decimals(x):
    """How many decimals x is written with, at 15 significant digits: 0.1 has 1, 20 has 0."""
    return max(0, -Decimal(f"{x:.15g}").as_tuple().exponent)


def multiple_below(x, step):
    """The multiple of step at or below x, with the multiples on the decimal grid."""
    places = decimals(step)
    k = np.floor(x / step)
    if round(float((k + 1) * step), places) <= x:
        k += 1
    elif round(float(k * step), places) > x:
        k -= 1
    return round(float(k * step), places)


def power_of_ten(exponent):
    return round(10.0 ** exponent, max(0, -exponent))


def stem_unit(width):
    exponent = int(np.ceil(np.log10(width)))
    while power_of_ten(exponent - 1) >= width:
        exponent -= 1
    while power_of_ten(exponent) < width:
        exponent += 1
    return power_of_ten(exponent)


def stem_and_leaf(x, width):
    unit = stem_unit(width)
    leaf_unit = round(unit / 10, decimals(unit) + 1)
    rows_per_stem = round(unit / width)
    # Each value's row, counted from 0 on its size, and its leaf. x is sorted.
    placed = [(bool(v < 0), round(multiple_below(abs(v), width) / width),
               round(multiple_below(abs(v), leaf_unit) / leaf_unit) % 10) for v in x]
    (low_negative, low_row, _), (high_negative, high_row, _) = placed[0], placed[-1]
    # From the lowest value's row to the highest's: the negative rows from the largest
    # size down to the stem -0, then the stem 0 and up.
    order = []
    if low_negative:
        order += [(True, r) for r in range(low_row, (high_row if high_negative else 0) - 1, -1)]
    if not high_negative:
        order += [(False, r) for r in range(0 if low_negative else low_row, high_row + 1)]
    leaves = {key: "" for key in order}
    for negative, row, leaf in placed:
        leaves[(negative, row)] += str(leaf)
    rows = []
    for (negative, row), text in leaves.items():
        stem = round(row // rows_per_stem * unit, decimals(unit))
        rows.append([len(text), -stem if negative else stem, text])
    return unit, pd.DataFrame(rows, columns=["frequency", "stem", "leaves"])


stems = {}
for name, x in series.items():
    width = stem_width(x)
    unit, stems[name] = stem_and_leaf(x, width)
    print(f"\n{name}: stem unit {unit:g}, stem width {width:g}")
    print(stems[name].to_string(index=False))

## What the result says

A stem-and-leaf display is a histogram made of digits. The stem is the tens digit and each leaf is the last digit of one delivery time. Each stem is split in two rows: leaves from 0 to 4 and leaves from 5 to 9.

For ZipEats the longest row is the second row of stem 20, with 22 orders from 25 to 29 minutes. For FeastNow it is the first row of stem 30, also with 22 orders, and for UrbanBite the first row of stem 40, with 15 orders.

The display also shows what the averages hide. ZipEats has one order on the last row, 58 minutes, after four empty rows. Every other ZipEats order took 31 minutes or less.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.